In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pycocotools timm einops
!git clone https://github.com/facebookresearch/detr.git
%cd detr

Cloning into 'detr'...
remote: Enumerating objects: 265, done.
remote: Total 265 (delta 0), reused 0 (delta 0), pack-reused 265 (from 1)
Receiving objects: 100% (265/265), 21.19 MiB | 36.41 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/kaggle/working/detr


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as funcy
import math

In [3]:
class Rounding(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x):
        return torch.round(x)
    @staticmethod
    def backward(ctx,grad_output):
        return grad_output

In [4]:
class Ternary(nn.Module):
    def __init__(self,inn,out,bias=True):
        super().__init__()
        self.inn=inn
        self.out=out
        self.weight=nn.Parameter(torch.empty(out,inn,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.weight,a=math.sqrt(5))
        self.s=nn.Parameter(torch.tensor(0.1,dtype=torch.float32))
        self.register_buffer('qweight',torch.zeros(out,inn,dtype=torch.int8))
        if bias:
            self.bias=nn.Parameter(torch.zeros(out,dtype=torch.float32))
        else:
            self.bias=None
        with torch.no_grad():
            self.s.data=torch.mean(torch.abs(self.weight)).to(torch.float32).clamp(min=1e-5)
    def forward(self,x):
        if self.training:
            sc=torch.mean(torch.abs(self.weight)).to(torch.float32).clamp(min=1e-5)
            wsc=self.weight/sc
            wcl=torch.clamp(wsc,-1.0,1.0)
            wter=Rounding.apply(wcl)
            ewe=sc*wter.float()
        else:
            ewe=self.s*self.qweight.float()
        return funcy.linear(x,ewe,self.bias)
    def pack_for_inference(self):
        with torch.no_grad():
            sc=torch.mean(torch.abs(self.weight)).to(torch.float32).clamp(min=1e-5)
            wsc=self.weight/sc
            wcl=torch.clamp(wsc,-1.0,1.0)
            self.qweight.copy_(torch.round(wcl).to(torch.int8))
            self.s.data=sc

In [5]:
def replace_with_scaled_ternary(mod):
    for name, module in list(mod.named_children()):
        if isinstance(module, nn.Linear):
            new_module = Ternary(
                module.in_features,
                module.out_features,
                bias=module.bias is not None
            )
            new_module.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_module.bias.data.copy_(module.bias.data.to(torch.float32))
            setattr(mod, name, new_module)
        else:
            replace_with_scaled_ternary(module)
    return mod

In [6]:
import torch
from argparse import Namespace
from models.detr import build   # from the cloned repo

# Complete args object that satisfies EVERY requirement in build(), build_backbone(), build_transformer()
args = Namespace(
    dataset_file="coco",
    backbone="resnet50",           # change to "resnet101" later if you want
    num_queries=100,
    aux_loss=True,
    device="cuda",
    hidden_dim=256,
    dropout=0.1,
    nheads=8,
    enc_layers=6,
    dec_layers=6,
    dim_feedforward=2048,
    position_embedding="sine",
    dilation=False,
    normalize_before=False,
    lr_backbone=1e-5,              # must be >0 so backbone is trained
    masks=False,
    pre_norm=True,
    set_cost_class=1,
    set_cost_bbox=5,
    set_cost_giou=2,
    bbox_loss_coef=5,
    giou_loss_coef=2,eos_coef=0.1,# not using panoptic segmentation
)

# Build the full model + criterion + postprocessors
mod, criterion, postprocessors = build(args)

# Apply 1.58-bit ternary QAT to ALL linear layers (FFN + attention projections)
mod = replace_with_scaled_ternary(mod)
mod = mod.cuda()

print("✅ DETR-R50 successfully built with ScaledTernaryLinear (1.58-bit + shared FP16 scale)!")
print(f"   Total parameters: {sum(p.numel() for p in mod.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in mod.parameters() if p.requires_grad):,}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]


✅ DETR-R50 successfully built with ScaledTernaryLinear (1.58-bit + shared FP16 scale)!
   Total parameters: 41,525,326
   Trainable parameters: 41,302,926


In [7]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [8]:
import datasets
from torch.utils.data import Subset
# Standard COCO loader from DETR repo
dataset_train = datasets.build_dataset(image_set='train', args=type('args', (), {'dataset_file': 'coco', 'coco_path': '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017','masks':False})())

# Optional: use only first 10,000 images for fast QAT demo (full train=118k is too slow on Kaggle)
dataset_train = Subset(dataset_train, range(10000))   # ← change to len(dataset_train) for full (if you have time)

sampler = torch.utils.data.RandomSampler(dataset_train)
data_loader_train = torch.utils.data.DataLoader(
    dataset_train, batch_size=1, sampler=sampler,  # small batch for VRAM
    collate_fn=collate_fn, num_workers=0
)
dataset_val = datasets.build_dataset(image_set='val', args=type('args', (), {'dataset_file': 'coco', 'coco_path': '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017','masks':False})())
data_loader_val = torch.utils.data.DataLoader(dataset_val, batch_size=1, shuffle=False, collate_fn=collate_fn)

loading annotations into memory...
Done (t=16.75s)
creating index...
index created!
loading annotations into memory...
Done (t=0.87s)
creating index...
index created!


In [9]:
optimizer = torch.optim.AdamW(mod.parameters(), lr=1e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_23/302039315.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [10]:
import torch
torch.cuda.empty_cache()

In [11]:
for epoch in range(20):   # increase later if you want
    mod.train()
    running_loss = 0.0
    
    for samples, targets in data_loader_train:
        samples = [s.cuda() for s in samples]
        targets = [{k: v.cuda() for k, v in t.items()} for t in targets]
        
        with torch.cuda.amp.autocast():
            outputs = mod(samples)
            loss_dict = criterion(outputs, targets)          # ← use the separate criterion!
            loss = sum(loss_dict.values())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(data_loader_train)
    print(f"Epoch {epoch+1:2d} | Avg Loss: {avg_loss:.4f}")
    if (epoch + 1) % 5 == 0:
        torch.save(mod.state_dict(), f"/kaggle/working/detr_1.58bit_epoch{epoch+1}.pth")

/tmp/ipykernel_23/309496705.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


AssertionError: 

In [ ]:
mod.eval()
for module in mod.modules():
    if isinstance(module, ScaledTernaryLinear):
        module.pack_for_inference()

torch.save(mod.state_dict(), "/kaggle/working/detr_1.58bit_packed_final.pth")